# VoxForge 🎤 — AI Voice Conversion Studio

**VoxForge** is a streamlined AI voice conversion tool built on RVC (Retrieval-based Voice Conversion).

### Features
- 🎙️ **Speech Conversion** — Convert text-to-speech using Edge TTS + RVC voice models (one-click and multi-step workflows)
- 📤 **Model Upload** — Upload your own `.pth` / `.index` RVC voice models
- ⚙️ **Settings & Presets** — Save and load UI configuration presets
- 🔄 **Audio Format Converter** — Convert audio files between WAV, MP3, FLAC, OGG, and M4A

### Instructions
1. Run cells **1 → 4** in order.
2. *(Optional)* Upload a voice model in cell **3** before launching.
3. Cell **4** will start the web UI and print a public link.

In [ ]:
# @title 1: Clone repository
from IPython.display import clear_output

REPO_URL = "https://github.com/BeastTamil/VOXFORGE.git"  # @param {type:"string"}
BRANCH   = "main"  # @param {type:"string"}

%cd /content
!rm -rf /content/VOXFORGE
!git clone --branch $BRANCH --depth 1 $REPO_URL /content/VOXFORGE
%cd /content/VOXFORGE
clear_output()
print("Repository cloned.")

In [ ]:
# @title 2: Install dependencies
import os
from IPython.display import clear_output

!apt-get install -y python3-dev unzip ffmpeg -q
!curl -LsSf https://astral.sh/uv/install.sh | sh

os.environ["PATH"] = f"{os.path.expanduser('~/.local/bin')}:{os.environ['PATH']}"
os.environ["URVC_CONSOLE_LOG_LEVEL"] = "WARNING"

!uv sync --directory /content/VOXFORGE --prerelease if-necessary-or-explicit -q

clear_output()
print("Dependencies installed.")

In [ ]:
# @title 3 (optional): Upload voice model
MODEL_NAME = "My Voice Model"  # @param {type:"string"}

from pathlib import Path
from google.colab import files as _files

print("Upload your .pth file (and optional .index file):")
uploaded = _files.upload()

if uploaded:
    dest = Path(f"/content/VOXFORGE/models/voice/{MODEL_NAME}")
    dest.mkdir(parents=True, exist_ok=True)
    for filename, data in uploaded.items():
        (dest / filename).write_bytes(data)
        print(f"Saved: {filename}")
    print(f"\nModel '{MODEL_NAME}' is ready.")
else:
    print("Skipped. Use the Models > Upload tab in the UI instead.")

In [ ]:
# @title 4: Launch VoxForge
import os, time

method      = "gradio"   # @param ["gradio", "ngrok", "cloudflared"]
ngrok_token = ""         # @param {type:"string"}

os.environ["PATH"] = f"{os.path.expanduser('~/.local/bin')}:{os.environ['PATH']}"
os.environ["URVC_CONSOLE_LOG_LEVEL"] = "WARNING"
os.environ["GRADIO_TEMP_DIR"] = "/content/VOXFORGE/temp"

UV  = "MPLBACKEND=Agg uv run --directory /content/VOXFORGE --prerelease if-necessary-or-explicit"
APP = "/content/VOXFORGE/src/ultimate_rvc/web/main.py"

if method == "gradio":
    !{UV} {APP} --share
elif method == "ngrok":
    !pip install -q pyngrok
    from pyngrok import ngrok as _ngrok
    _ngrok.set_auth_token(ngrok_token)
    _ngrok.kill()
    tunnel = _ngrok.connect(7860)
    print(f"Ngrok URL: {tunnel.public_url}")
    !{UV} {APP} --listen-port 7860
elif method == "cloudflared":
    !curl -LO https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -q
    !dpkg -i cloudflared-linux-amd64.deb -q
    !rm -rf nohup.out
    !nohup cloudflared tunnel --url localhost:7860 &
    time.sleep(10)
    url = !grep -oE 'https://[a-zA-Z0-9.-]+\.trycloudflare\.com' nohup.out
    print(f"URL: {url}")
    !{UV} {APP} --listen-port 7860